In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("env", "dev", "Ambiente (dev | prod)")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog_base", "catalog_pr_db")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsdvbtdev01")


env = dbutils.widgets.get("env").strip().lower()
container = dbutils.widgets.get("container")
catalogo = f"{dbutils.widgets.get('catalog_base')}_{env}"
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
 
assert env in ("dev", "prod"), f"env inválido: {env}"
 
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/drivers.csv"
print(f"env={env} | catalogo={catalogo} | ruta={ruta}")


In [0]:
df_drivers = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
df_drivers.display()

In [0]:
driver_schema = StructType(fields=[StructField("driverId", IntegerType(), True),
                                   StructField("driverRef", StringType(), False),
                                     StructField("number", IntegerType(), True),
                                     StructField("code", StringType(), True),
                                     StructField("forename", StringType(), True),
                                     StructField("surname", StringType(), True),
                                     StructField("dob", DateType(), True),
                                     StructField("nationality", StringType(), True)
])

In [0]:
driver_schema

In [0]:
df_drivers_final = spark.read\
.option('header', True)\
.schema(driver_schema)\
.csv(ruta)

In [0]:
drivers_selected_df = df_drivers_final.select(col("driverId").alias('driver_id'), 
                                                col("driverRef"), 
                                                col("number").alias('number_driver'), 
                                                col("code"),
                                                col("forename"), 
                                                col("surname"), 
                                                col("dob"),
                                                col("nationality"))


                

In [0]:
drivers_selected_df

In [0]:
drivers_final_df = drivers_selected_df.withColumn("ingestion_date", current_timestamp())

In [0]:
drivers_final_df.display()

In [0]:
print(f"catalogo = {catalogo}")
print(f"esquema = {esquema}")

In [0]:
spark.sql(f"USE CATALOG `{catalogo}`")
spark.sql(f"USE SCHEMA `{esquema}`")

In [0]:
print(f"current_catalog = {spark.sql('SELECT current_catalog()').collect()[0][0]}")
print(f"current_schema = {spark.sql('SELECT current_schema()').collect()[0][0]}")


In [0]:
drivers_final_df.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema}.drivers")

In [0]:
%sql
--select * from catalog_pr_db_dev.bronze.drivers